# Capability: inference (V1 Stub)

The `inference` capability is intended for AI-powered extraction when a field cannot be resolved by rule-based methods alone.

> **V1 limitation:** The current `inference` capability is a **stub**. It returns deterministic placeholder values. Real LLM-backed inference is planned for **V2** (see issues #50, #51).
>
> In V1, the stub demonstrates the *plumbing* — you can see inference candidates appear in the evidence chain, the reconciler grades them, and unresolved fields are reported. The actual AI call is out of scope for V1.

In [ ]:
from pydantic import BaseModel, Field

import paxman
import paxman.contract.adapters.pydantic  # noqa: F401


class ComplexField(BaseModel):
    description: str = Field(..., description="Free-text description")


sample = "The vendor is ACME Corp, located in Kuala Lumpur, established 2010."

result = paxman.normalize(sample, ComplexField)
print(f"Status: {result.status.name}")
print(f"Data:   {result.normalized_data}")
print(f"Unresolved: {result.unresolved_fields}")
print()
print("(The inference stub returns deterministic values; real LLM calls land in V2.)")

In [ ]:
# Show what happens when no rule-based capability can resolve a field
class Ambiguous(BaseModel):
    sentiment: str = Field(..., description="Positive, negative, or neutral")


result_ambig = paxman.normalize(
    "The product quality is excellent, delivery was on time.", Ambiguous
)
print(f"Status: {result_ambig.status.name}")
print(f"Unresolved: {result_ambig.unresolved_fields}")
print()
print("(The inference stub cannot resolve this either — a real LLM call would be needed.)")

## V1 Stub vs V2 Real Inference

| Aspect | V1 (current) | V2 (planned) |
|---|---|---|
| Provider | Deterministic stub | OpenAI, Anthropic, local models |
| Cost model | `max_total_cost_usd` (no-op) | Real token counting |
| Confidence | Fixed `MEDIUM` | LLM-calibrated confidence |
| Evidence | Placeholder ref | Actual prompt + response |

**What you CAN do in V1:** Register a custom capability that calls an LLM via `paxman.register_capability()` (see notebook 04 — the pattern is identical).

## Try it yourself

- Register a custom capability that delegates to a local LLM (via `openai` SDK or similar).
- Compare the evidence chain when inference is the *only* capability that matches a field.
- Check the `diagnostics` list — does the planner warn you when only inference can resolve a field?